In [ ]:
from langgraph.graph import StateGraph, START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import operator

load_dotenv()

In [ ]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

In [ ]:
class feedbackSchema(BaseModel):
    feedback: str = Field(description="feedback on essay written by student.")
    score:int = Field(description="score out of 10",ge=0,le=10)

structured_model = model.with_structured_output(feedbackSchema)

In [ ]:
class EssayState(TypedDict):
    essay: str
    
    COT_feedback: str
    DOT_feedback: str
    lang_feedback: str
    final_feedback:str

    individual_scores:Annotated[list[int],operator.add]
    avg_score:int
    

In [ ]:
def calc_COT(state:EssayState):
    prompt = f"Evaluate the clarity of thought of the essay and provide the feedback and assign a score out of 10. essay: {state['essay']}"
    ans = structured_model.invoke(prompt)
    return {"COT_feedback":ans.feedback,"individual_scores":[ans.score]}

def calc_DOA(state:EssayState):
    prompt = f"Evaluate the Depth of analaysis of the essay and provide the feedback and assign a score out of 10. essay: {state['essay']}"
    ans = structured_model.invoke(prompt)
    return {"DOT_feedback":ans.feedback,"individual_scores":[ans.score]}

def calc_lang_quality(state:EssayState):
    prompt = f"Evaluate the language quality of the essay and provide the feedback and assign a score out of 10. essay: {state['essay']}"
    ans = structured_model.invoke(prompt)
    return {"lang_feedback":ans.feedback,"individual_scores":[ans.score]}

def get_final_feedback(state:EssayState):
    prompt = f"Based on following feedback, create a summarized feedback \n language quality feedback: {state['lang_feedback']} \n clarity of thoughts: {state['COT_feedback']} \n depth of analysis: {state['DOT_feedback']}"
    
    ans = model.invoke(prompt).content

    # avg calculation
    avg_score = sum(state["individual_scores"])/len(state["individual_scores"])

    return {"final_feedback":ans,"avg_score":avg_score}


In [ ]:
graph = StateGraph(EssayState)

graph.add_node("calc_COT",calc_COT)
graph.add_node("calc_DOA",calc_DOA)
graph.add_node("calc_lang_quality",calc_lang_quality)
graph.add_node("get_final_feedback",get_final_feedback)

graph.add_edge(START,"calc_COT")
graph.add_edge(START,"calc_DOA")
graph.add_edge(START,"calc_lang_quality")

graph.add_edge("calc_COT","get_final_feedback")
graph.add_edge("calc_DOA","get_final_feedback")
graph.add_edge("calc_lang_quality","get_final_feedback")

graph.add_edge("get_final_feedback",END)

workflow = graph.compile()

In [ ]:
sample_essay = """# My Favorite Season

My favorite season is summer because it is very good and I like it very much. Summer is the best season than all the others because everybody likes going outside and doing fun. I think it is the most happiest time in the year.

In summer the weather are hot and sunny. Peoples goes to the beach, swimming pools, and parks. I likes eating ice cream almost every days because it make me happy. Sometimes the sun is too much hot but it is okay because we can drink cold waters and sit under a tree.

My family always go on a trip in summer vacation. Last year we goes to my uncle village. It was very fun because there had many trees and animals. Me and my cousins was playing cricket every morning. We also catch fishes in the river even though we don't catch many. At night we was telling scary stories and everybody laugh very loud.

Summer also means no school for many weeks. This are the best part because students don't have to waking up early. I can sleep until 10 o'clock and then I eat breakfast. My mother always tell me to help in the house, but sometimes I forgetted because I was busy watching cartoons.

Some people says summer is bad because it is too hot. I don't agrees with them. Every season have some problems. Winter is cold and rainy season have too much rain. So summer is still more better for me because I can play outside and wear my favorite clothes.

In conclusion, summer is my favorite season and I will always choose it. It have many good things and only little bad things. I hope every month was summer because life would be more happier and everyone can enjoying outside together. That is why summer are the greatest season in the whole world.
"""

initial_state = {"essay":sample_essay}

final_state = workflow.invoke(initial_state)
print(final_state)